# Non-AI Object Counting — OpenCV V2

## Matrix Smart Technologies AI Engineer Assessment

**Approach:** Classical computer vision using image preprocessing,
foreground extraction, distance transform, watershed segmentation,
and region filtering.

### Execution
1. Mount Google Drive.
2. Confirm the dataset folders are available under the project root.
3. Execute the notebook from top to bottom.
4. Review the generated prediction CSV, object details, masks, and overlays.

This notebook does not use a trained deep-learning detection model.


## 1. Dependencies and imports

In [ ]:
from google.colab import drive
drive.mount('/content/gdrive')

In [ ]:

# ============================================================
# PHASE 2: ACCURACY-OPTIMIZED OPENCV
# CELL 1: ENVIRONMENT SETUP
# ============================================================

!pip -q install opencv-python-headless scikit-image scipy pandas matplotlib

import os
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pathlib import Path
from skimage.feature import peak_local_max
from skimage.segmentation import watershed
from skimage.measure import regionprops
from scipy import ndimage as ndi


print("OpenCV version:", cv2.__version__)
print("NumPy version:", np.__version__)
print("Environment ready!")


## 2. Dataset discovery and configuration

In [ ]:

# ============================================================
# CELL 2: DATASET DISCOVERY AND CONFIGURATION
# ============================================================

PROJECT_ROOT = Path(
    "/content/gdrive/MyDrive/Matrix_AI_Assessment"
)

DATASET_NAMES = [
    "ScrewAndBolt_20240713",
    "Screws_2024_07_15"
]

OUTPUT_DIR = Path("/content/gdrive/MyDrive/Matrix_AI_Assessment/Matrix_AI_Assessment_Submission/Non_AI/outputs/run")
OVERLAY_DIR = OUTPUT_DIR / "overlays"
MASK_DIR = OUTPUT_DIR / "masks"
REVIEW_DIR = OUTPUT_DIR / "review_sheets"

for folder in [OUTPUT_DIR, OVERLAY_DIR, MASK_DIR, REVIEW_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp"}

# Parameters calibrated to the two image resolutions
CONFIG = {
    "min_distance_scale": 0.0267,
    "peak_threshold_rel": 0.15,
    "min_area_ratio": 0.000005,
    "min_area_absolute": 50,
    "background_kernel_scale": 0.04,
    "morphology_open": 3,
    "morphology_close": 5
}

# Find dataset directories
dataset_paths = {}

for current_root, dirs, files in os.walk(PROJECT_ROOT):
    for dataset_name in DATASET_NAMES:
        if os.path.basename(current_root) == dataset_name:
            dataset_paths[dataset_name] = Path(current_root)

    # Avoid scanning previous generated results
    dirs[:] = [
        d for d in dirs
        if d not in {
            "opencv_accuracy_v2",
            "opencv_results",
            ".ipynb_checkpoints"
        }
    ]

print("\nDATASET DISCOVERY")
print("=" * 55)

for name in DATASET_NAMES:
    if name in dataset_paths:
        print("FOUND:", name, "->", dataset_paths[name])
    else:
        print("NOT FOUND:", name)

if len(dataset_paths) != 2:
    raise FileNotFoundError(
        "Both dataset folders must exist under PROJECT_ROOT. "
        "If they are elsewhere, update PROJECT_ROOT in this cell."
    )

print("\nConfiguration:")
print(CONFIG)


## 3. Image processing and watershed functions

In [ ]:

# ============================================================
# CELL 3: ACCURACY-OPTIMIZED COUNTING FUNCTIONS
# ============================================================

def preprocess_image(image):
    """
    Convert image to grayscale and reduce noise.
    """
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)

    # Small noise reduction
    gray = cv2.GaussianBlur(gray, (3, 3), 0)

    return gray


def create_foreground_mask(gray):
    """
    Estimate background illumination and extract dark objects.
    """

    h, w = gray.shape
    min_dim = min(h, w)

    # Large kernel estimates local background
    kernel_size = max(
        15,
        int(min_dim * CONFIG["background_kernel_scale"])
    )

    if kernel_size % 2 == 0:
        kernel_size += 1

    background_kernel = cv2.getStructuringElement(
        cv2.MORPH_ELLIPSE,
        (kernel_size, kernel_size)
    )

    # Morphological closing estimates the background
    background = cv2.morphologyEx(
        gray,
        cv2.MORPH_CLOSE,
        background_kernel
    )

    # Dark objects become bright in the difference image
    dark_response = cv2.subtract(background, gray)

    # Otsu thresholding
    threshold_value, mask = cv2.threshold(
        dark_response,
        0,
        255,
        cv2.THRESH_BINARY + cv2.THRESH_OTSU
    )

    # Remove tiny isolated noise
    open_kernel = cv2.getStructuringElement(
        cv2.MORPH_ELLIPSE,
        (
            CONFIG["morphology_open"],
            CONFIG["morphology_open"]
        )
    )

    mask = cv2.morphologyEx(
        mask,
        cv2.MORPH_OPEN,
        open_kernel
    )

    # Connect small gaps within individual objects
    close_kernel = cv2.getStructuringElement(
        cv2.MORPH_ELLIPSE,
        (
            CONFIG["morphology_close"],
            CONFIG["morphology_close"]
        )
    )

    mask = cv2.morphologyEx(
        mask,
        cv2.MORPH_CLOSE,
        close_kernel
    )

    return mask, dark_response, threshold_value


def remove_large_border_artifacts(mask):
    """
    Remove unusually large connected components touching
    image boundaries. Preserve smaller objects near edges.
    """

    h, w = mask.shape
    total_pixels = h * w

    count, labels, stats, centroids = (
        cv2.connectedComponentsWithStats(
            mask,
            connectivity=8
        )
    )

    cleaned = mask.copy()

    for i in range(1, count):

        x = stats[i, cv2.CC_STAT_LEFT]
        y = stats[i, cv2.CC_STAT_TOP]
        bw = stats[i, cv2.CC_STAT_WIDTH]
        bh = stats[i, cv2.CC_STAT_HEIGHT]
        area = stats[i, cv2.CC_STAT_AREA]

        touches_border = (
            x == 0 or y == 0 or
            x + bw >= w or y + bh >= h
        )

        # Only remove large border-connected artifacts
        if touches_border and area > total_pixels * 0.0005:
            cleaned[labels == i] = 0

    return cleaned


def generate_watershed_markers(mask, min_distance):
    """
    Generate spatially separated object-center markers.
    """

    binary = mask > 0

    # Distance transform
    distance = cv2.distanceTransform(
        mask,
        cv2.DIST_L2,
        5
    )

    # Local maxima identify potential object centers
    coordinates = peak_local_max(
        distance,
        min_distance=min_distance,
        threshold_rel=CONFIG["peak_threshold_rel"],
        labels=binary,
        exclude_border=False
    )

    markers = np.zeros(mask.shape, dtype=np.int32)

    for idx, (y, x) in enumerate(coordinates, start=1):
        markers[y, x] = idx

    # Ensure watershed markers are properly labeled
    markers = ndi.label(markers > 0)[0].astype(np.int32)

    return markers, distance, len(coordinates)


def segment_objects(image, mask, markers):
    """
    Separate touching objects using watershed segmentation.
    """

    # Watershed operates on the negative distance transform
    distance = cv2.distanceTransform(
        mask,
        cv2.DIST_L2,
        5
    )

    labels = watershed(
        -distance,
        markers,
        mask=(mask > 0),
        watershed_line=False
    )

    return labels


def validate_regions(labels, image_shape):
    """
    Filter tiny regions and highly implausible regions.
    """

    h, w = image_shape
    total_pixels = h * w

    min_area = max(
        CONFIG["min_area_absolute"],
        int(total_pixels * CONFIG["min_area_ratio"])
    )

    properties = regionprops(labels)

    accepted_regions = []

    for region in properties:

        area = region.area

        if area < min_area:
            continue

        min_row, min_col, max_row, max_col = region.bbox

        box_height = max_row - min_row
        box_width = max_col - min_col

        if box_height == 0 or box_width == 0:
            continue

        aspect_ratio = max(
            box_height / box_width,
            box_width / box_height
        )

        # Conservative rejection of extreme noise shapes
        if aspect_ratio > 10:
            continue

        if region.solidity < 0.10:
            continue

        accepted_regions.append({
            "label": int(region.label),
            "area": int(area),
            "bbox": (
                int(min_col),
                int(min_row),
                int(max_col),
                int(max_row)
            ),
            "centroid": region.centroid,
            "aspect_ratio": float(aspect_ratio),
            "solidity": float(region.solidity)
        })

    return accepted_regions, min_area


def create_overlay(image, accepted_regions):
    """
    Draw bounding boxes and object IDs.
    """

    overlay = image.copy()

    for idx, region in enumerate(accepted_regions, start=1):

        x1, y1, x2, y2 = region["bbox"]

        cv2.rectangle(
            overlay,
            (x1, y1),
            (x2, y2),
            (0, 255, 0),
            3
        )

        cv2.putText(
            overlay,
            str(idx),
            (x1, max(25, y1 - 8)),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.7,
            (0, 0, 255),
            2
        )

    cv2.putText(
        overlay,
        f"Count: {len(accepted_regions)}",
        (30, 60),
        cv2.FONT_HERSHEY_SIMPLEX,
        1.5,
        (0, 0, 255),
        3
    )

    return overlay


print("All processing functions initialized successfully.")


## 4. Full dataset inference

In [ ]:

# ============================================================
# CELL 4: BATCH PROCESS BOTH DATASETS
# ============================================================

prediction_results = []
object_details = []
review_images = []

for dataset_name, dataset_path in dataset_paths.items():

    print("\n" + "=" * 65)
    print("PROCESSING DATASET:", dataset_name)
    print("=" * 65)

    image_files = sorted([
        p for p in dataset_path.rglob("*")
        if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS
    ])

    print("Images found:", len(image_files))

    dataset_overlay_dir = OVERLAY_DIR / dataset_name
    dataset_mask_dir = MASK_DIR / dataset_name

    dataset_overlay_dir.mkdir(parents=True, exist_ok=True)
    dataset_mask_dir.mkdir(parents=True, exist_ok=True)

    for image_path in image_files:

        filename = image_path.name

        try:
            image = cv2.imread(str(image_path))

            if image is None:
                raise ValueError("Unable to read image")

            h, w = image.shape[:2]
            min_dim = min(h, w)

            # Resolution-scaled minimum distance
            min_distance = max(
                5,
                int(round(
                    min_dim * CONFIG["min_distance_scale"]
                ))
            )

            # Step 1: Preprocessing
            gray = preprocess_image(image)

            # Step 2: Foreground extraction
            mask, dark_response, otsu_value = (
                create_foreground_mask(gray)
            )

            # Step 3: Remove large border artifacts
            mask = remove_large_border_artifacts(mask)

            # Step 4: Generate markers
            markers, distance, marker_count = (
                generate_watershed_markers(
                    mask,
                    min_distance
                )
            )

            # Step 5: Watershed segmentation
            labels = segment_objects(
                image,
                mask,
                markers
            )

            # Step 6: Validate regions
            accepted_regions, min_area = validate_regions(
                labels,
                (h, w)
            )

            predicted_count = len(accepted_regions)

            # Step 7: Create visualization
            overlay = create_overlay(
                image,
                accepted_regions
            )

            # Save overlay and mask
            overlay_path = dataset_overlay_dir / (
                image_path.stem + "_overlay.jpg"
            )

            mask_path = dataset_mask_dir / (
                image_path.stem + "_mask.png"
            )

            cv2.imwrite(str(overlay_path), overlay)
            cv2.imwrite(str(mask_path), mask)

            # Save object-level measurements
            for obj_id, region in enumerate(
                accepted_regions,
                start=1
            ):

                x1, y1, x2, y2 = region["bbox"]

                object_details.append({
                    "dataset": dataset_name,
                    "filename": filename,
                    "object_id": obj_id,
                    "area": region["area"],
                    "x1": x1,
                    "y1": y1,
                    "x2": x2,
                    "y2": y2,
                    "aspect_ratio": region["aspect_ratio"],
                    "solidity": region["solidity"]
                })

            prediction_results.append({
                "dataset": dataset_name,
                "filename": filename,
                "image_width": w,
                "image_height": h,
                "otsu_threshold": float(otsu_value),
                "min_distance": min_distance,
                "markers": marker_count,
                "raw_regions": len(np.unique(labels)) - 1,
                "minimum_area": min_area,
                "predicted_count": predicted_count,
                "status": "success"
            })

            # Store smaller copies for contact sheets
            max_width = 650
            scale = min(1.0, max_width / w)

            small_overlay = cv2.resize(
                overlay,
                (
                    int(w * scale),
                    int(h * scale)
                )
            )

            review_images.append({
                "dataset": dataset_name,
                "filename": filename,
                "count": predicted_count,
                "image": cv2.cvtColor(
                    small_overlay,
                    cv2.COLOR_BGR2RGB
                )
            })

            print(
                f"{filename}: {predicted_count} objects | "
                f"Markers: {marker_count} | "
                f"Min distance: {min_distance}"
            )

        except Exception as e:

            print("ERROR:", filename, str(e))

            prediction_results.append({
                "dataset": dataset_name,
                "filename": filename,
                "predicted_count": np.nan,
                "status": "failed: " + str(e)
            })


# ============================================================
# SAVE RESULTS
# ============================================================

predictions_df = pd.DataFrame(prediction_results)
objects_df = pd.DataFrame(object_details)

prediction_csv = OUTPUT_DIR / "opencv_predictions_v2.csv"
objects_csv = OUTPUT_DIR / "opencv_object_details_v2.csv"

predictions_df.to_csv(prediction_csv, index=False)
objects_df.to_csv(objects_csv, index=False)

print("\n" + "=" * 65)
print("BATCH PROCESSING COMPLETED")
print("=" * 65)

print("Images processed:", len(predictions_df))
print("Successful:", (predictions_df["status"] == "success").sum())
print("Failed:", (predictions_df["status"] != "success").sum())
print("Total detected objects:", predictions_df["predicted_count"].sum())

print("\nPredictions saved:", prediction_csv)
print("Object details saved:", objects_csv)

display(predictions_df)


## 5. Visual review sheets

In [ ]:

# ============================================================
# CELL 5: VISUAL REVIEW SHEETS
# ============================================================

for dataset_name in DATASET_NAMES:

    selected = [
        item for item in review_images
        if item["dataset"] == dataset_name
    ]

    if not selected:
        continue

    images_per_page = 6

    for page_start in range(
        0, len(selected), images_per_page
    ):

        batch = selected[
            page_start:page_start + images_per_page
        ]

        fig, axes = plt.subplots(
            2, 3,
            figsize=(18, 11)
        )

        axes = axes.flatten()

        for ax, item in zip(axes, batch):

            ax.imshow(item["image"])

            ax.set_title(
                f'{item["filename"]}\n'
                f'Predicted objects: {item["count"]}',
                fontsize=10
            )

            ax.axis("off")

        for ax in axes[len(batch):]:
            ax.axis("off")

        plt.suptitle(
            f"{dataset_name} - Accuracy Review "
            f"Page {page_start // images_per_page + 1}",
            fontsize=16
        )

        plt.tight_layout()

        output_path = REVIEW_DIR / (
            f"{dataset_name}_page_"
            f"{page_start // images_per_page + 1}.jpg"
        )

        plt.savefig(
            output_path,
            dpi=150,
            bbox_inches="tight"
        )

        plt.show()
        plt.close()

print("All review sheets generated.")
print("Review folder:", REVIEW_DIR)


## Output artifacts

The inference workflow generates prediction records, object-level
measurements, annotated images, binary masks, and visual review sheets.

The verified evaluation results are provided separately in the
`evaluation` directory.
